# MyLake - Conexión al Data Lake

Este notebook está pre-configurado para conectarse a los datos de MyLake.

## Configuración disponible:
- **PostgreSQL**: Base de datos transaccional (usuarios, catálogo)
- **S3/RustFS**: Almacenamiento de objetos para datos
- **Parquet**: Formatos de datos para lakehouse

In [ ]:
# Configuración de conexión al Lake
import os

# PostgreSQL
LAKE_DB = {
    'host': 'postgres',
    'port': 5432,
    'database': 'mylake',
    'user': 'admin',
    'password': 'change-me-locally'
}

# S3/RustFS
LAKE_S3 = {
    'endpoint': 'http://rustfs:9000',
    'access_key': 'mylake-access',
    'secret_key': 'mylake-secret-key',
    'bucket': 'lakehouse'
}

print("✓ Configuración cargada")
print(f"  PostgreSQL: {LAKE_DB['host']}:{LAKE_DB['port']}/{LAKE_DB['database']}")
print(f"  S3: {LAKE_S3['endpoint']}")

In [ ]:
# Inicializar Spark con soporte para PostgreSQL y S3
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MyLake-Notebook") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.1,org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.hadoop.fs.s3a.endpoint", LAKE_S3['endpoint']) \
    .config("spark.hadoop.fs.s3a.access.key", LAKE_S3['access_key']) \
    .config("spark.hadoop.fs.s3a.secret.key", LAKE_S3['secret_key']) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print(f"✓ Spark Session creada: {spark.version}")
print(f"  App: {spark.sparkContext.appName}")
print(f"  Cores: {spark.sparkContext.defaultParallelism}")

## 📊 Explorar Metadatos del Lake

In [ ]:
# Leer tablas del catálogo desde PostgreSQL
jdbc_url = f"jdbc:postgresql://{LAKE_DB['host']}:{LAKE_DB['port']}/{LAKE_DB['database']}"
properties = {
    'user': LAKE_DB['user'],
    'password': LAKE_DB['password'],
    'driver': 'org.postgresql.Driver'
}

# Tablas disponibles
tables_df = spark.read.jdbc(
    url=jdbc_url,
    table="information_schema.tables",
    properties=properties
).filter("table_schema IN ('auth_mgmt', 'ducklake_catalog')")

print("Tablas disponibles en el Lake:")
tables_df.select('table_schema', 'table_name', 'table_type').show(truncate=False)

## 👥 Ejemplo: Leer Tabla de Usuarios

In [ ]:
# Leer usuarios desde PostgreSQL
users_df = spark.read.jdbc(
    url=jdbc_url,
    table='auth_mgmt.users',
    properties=properties
)

print(f"Usuarios registrados: {users_df.count()}")
users_df.select('id', 'username', 'role', 'created_at').show(truncate=False)

## 💾 Trabajar con Datos (Lakehouse Pattern)

In [ ]:
# Crear datos de ejemplo y guardar en formato Lakehouse (Parquet en S3)
from pyspark.sql import Row
from datetime import datetime

# Datos de ejemplo
data = [
    Row(date=datetime(2024, 1, 1), producto="Laptop", categoria="Electrónica", ventas=1500, cantidad=10),
    Row(date=datetime(2024, 1, 2), producto="Mouse", categoria="Electrónica", ventas=200, cantidad=20),
    Row(date=datetime(2024, 1, 3), producto="Teclado", categoria="Electrónica", ventas=400, cantidad=15),
    Row(date=datetime(2024, 1, 4), producto="Monitor", categoria="Electrónica", ventas=800, cantidad=8),
    Row(date=datetime(2024, 1, 5), producto="Silla", categoria="Muebles", ventas=300, cantidad=5),
]

ventas_df = spark.createDataFrame(data)
ventas_df.show()

In [ ]:
# Guardar en formato Parquet (local por ahora - S3 cuando esté configurado)
output_path = "/home/jovyan/work/ventas"

ventas_df.write \
    .mode("overwrite") \
    .partitionBy("categoria") \
    .parquet(output_path)

print(f"✓ Datos guardados en: {output_path}")

# Ver archivos creados
import os
for root, dirs, files in os.walk(output_path):
    for f in files:
        filepath = os.path.join(root, f)
        size = os.path.getsize(filepath)
        print(f"  {filepath} ({size} bytes)")

In [ ]:
# Leer de vuelta y analizar
df_read = spark.read.parquet(output_path)

# Análisis con SQL
df_read.createOrReplaceTempView("ventas")

print("Ventas por categoría:")
spark.sql("")"
    SELECT categoria, SUM(ventas) as total_ventas, SUM(cantidad) as total_items
    FROM ventas
    GROUP BY categoria
""").show()

print("\nVentas por día:")
spark.sql("""
    SELECT date, SUM(ventas) as total
    FROM ventas
    GROUP BY date
    ORDER BY date
""").show()

## 🔗 Guardar en PostgreSQL (tablas del catálogo)

In [ ]:
# Guardar DataFrame en PostgreSQL
# Esto crea una tabla en el esquema ducklake_catalog

# Primero crear tabla de resumen
resumen_df = spark.sql("""
    SELECT categoria, SUM(ventas) as total_ventas, AVG(ventas) as promedio_ventas
    FROM ventas
    GROUP BY categoria
""").cache()

resumen_df.show()

# Guardar en PostgreSQL (descomenta para probar)
# resumen_df.write \
#     .mode("overwrite") \
#     .jdbc(url=jdbc_url, table="ducklake_catalog.ventas_resumen", properties=properties)
# print("✓ Tabla guardada en ducklake_catalog.ventas_resumen")

## 📁 Explorar el Workspace

In [ ]:
# Ver archivos en el workspace persistente
import glob

print("Archivos en el workspace (/home/jovyan/work):")
for f in glob.glob("/home/jovyan/work/**/*", recursive=True):
    if os.path.isfile(f):
        size = os.path.getsize(f)
        rel_path = f.replace("/home/jovyan/work/", "")
        print(f"  {rel_path} ({size} bytes)")